In [2]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
print(df.shape)

(7043, 21)


In [5]:
print(df.columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')


In [6]:
print(df.describe())

       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000


In [7]:
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [8]:
print(df.duplicated())

0       False
1       False
2       False
3       False
4       False
        ...  
7038    False
7039    False
7040    False
7041    False
7042    False
Length: 7043, dtype: bool


In [21]:
# 1. Make a copy of the dataframe
clean_df = df.copy()

# 2. Define the binary columns
features = [
    "Partner", "Dependents", "PhoneService", 
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling"
]

# 3. Standardise strings to lowercase across the selected columns
clean_df[features] = clean_df[features].apply(lambda x: x.str.lower())

# 4. Map "yes"/"no" directly to numerical 1/0
# This handles entries like "no internet service" by leaving them untouched 
# until you decide how to encode them.
clean_df[features] = clean_df[features].replace({"yes": 1, "no": 0})


In [22]:
print(clean_df.head(2))

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0       1          0       1            0   
1  5575-GNVDE    Male              0       0          0      34            1   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL              0  ...                0   
1                No             DSL              1  ...                1   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0           0           0               0  Month-to-month                1   
1           0           0               0        One year                0   

      PaymentMethod MonthlyCharges  TotalCharges Churn  
0  Electronic check          29.85         29.85    No  
1      Mailed check          56.95        1889.5    No  

[2 rows x 21 columns]


In [11]:
print(clean_df['Contract'].unique())

<ArrowStringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str


In [27]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import pandas as pd

# ==========================================
# 1. ORDINAL ENCODING (For Rank-Based Columns)
# ==========================================
# Encoding Contract because it has a logical progression/order
contract_encoder = OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']])
clean_df['Contract_encoder'] = contract_encoder.fit_transform(clean_df[['Contract']])


# ==========================================
# 2. ONE-HOT ENCODING (For Unordered Categories)
# ==========================================
# Initialize OneHotEncoder for columns with no inherent mathematical hierarchy
oh_encoder = OneHotEncoder(sparse_output=False)

# --- Encoding InternetService ---
internet_encoded = oh_encoder.fit_transform(clean_df[['InternetService']])
encoded_cols_internet = oh_encoder.get_feature_names_out(['InternetService'])
internet_df = pd.DataFrame(internet_encoded, columns=encoded_cols_internet, index=clean_df.index)

# --- Encoding MultipleLines ---
lines_encoded = oh_encoder.fit_transform(clean_df[['MultipleLines']])
encoded_cols_lines = oh_encoder.get_feature_names_out(['MultipleLines'])
lines_df = pd.DataFrame(lines_encoded, columns=encoded_cols_lines, index=clean_df.index)

# --- NEW: Encoding PaymentMethod ---
payment_encoded = oh_encoder.fit_transform(clean_df[['PaymentMethod']])
encoded_cols_payment = oh_encoder.get_feature_names_out(['PaymentMethod'])
payment_df = pd.DataFrame(payment_encoded, columns=encoded_cols_payment, index=clean_df.index)


# ==========================================
# 3. MERGING EVERYTHING BACK TOGETHER
# ==========================================
# Concatenate all the new binary columns into your main clean_df side-by-side
clean_df = pd.concat([clean_df, internet_df, lines_df, payment_df], axis=1)


# ==========================================
# 4. DATA CLEANUP (Ready for training)
# ==========================================
# Uncomment this line when you want to drop the old raw text columns 
# so they don't add redundant noise to your final machine learning model:
clean_df = clean_df.drop(columns=['Contract', 'InternetService', 'MultipleLines', 'PaymentMethod'])


In [28]:
print(clean_df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0       1          0       1            0   
1  5575-GNVDE    Male              0       0          0      34            1   
2  3668-QPYBK    Male              0       0          0       2            1   
3  7795-CFOCW    Male              0       0          0      45            0   
4  9237-HQITU  Female              0       0          0       2            1   

  OnlineSecurity OnlineBackup DeviceProtection  ... InternetService_DSL  \
0              0            1                0  ...                 1.0   
1              1            0                1  ...                 1.0   
2              1            1                0  ...                 1.0   
3              1            0                1  ...                 1.0   
4              0            0                0  ...                 0.0   

  InternetService_Fiber optic InternetService_No MultipleLines_No  \